<a href="https://colab.research.google.com/github/codeBMB/google_colabs/blob/BASIL/python_colabs/BASIL/First_weak_attempt_at_Vibecoding_BLAST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install bio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.4/321.4 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.1/52.1 kB 2.5 MB/s eta 0:00:00


### Using Bio.Blast to Search Genbank
Below is an example of performing a BLAST search using the NCBI WWW service, parsing the results, and downloading the corresponding sequences from Genbank.

In [ ]:
from Bio.Blast import NCBIWWW
from Bio.Blast import NCBIXML
from Bio import Entrez
from Bio import SeqIO

# Provide an email address for NCBI Entrez API usage
Entrez.email = "paul.craig@rit.edu"

# 1. Perform a BLAST search against Genbank using a protein sequence query
# Here we use a sample protein sequence (e.g., portion of insulin)
query_sequence = "MALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGERGFFYTPKTRREAEDLQVGQVELGGGPGAGSLQPLALEGSLQKRGIVEQCCTSICSLYQLENYCN"

print("Submitting BLAST protein search (blastp) to NCBI... This can take a moment.")
# For protein queries, we use 'blastp' and search the 'nr' (non-redundant) protein database
result_handle = NCBIWWW.qblast("blastp", "nr", query_sequence)

# 2. Parse the BLAST results
blast_record = NCBIXML.read(result_handle)
result_handle.close()

# 3. Retrieve and print the top match information
top_gi = None
if blast_record.alignments:
    first_alignment = blast_record.alignments[0]
    print(f"\nTop Alignment Title: {first_alignment.title}")
    print(f"Accession: {first_alignment.accession}")

    # Get the sequence ID/GI to download it
    top_gi = first_alignment.hit_id
    print(f"Hit ID: {top_gi}")
else:
    print("No alignments found.")

SyntaxError: invalid syntax (2705178483.py, line 11)

### Downloading the Sequence using Entrez
Using the sequence identifier obtained from the BLAST search (or any known accession number), we can retrieve the complete sequence record from Genbank.

In [ ]:
if top_gi:
    print(f"Downloading sequence for {top_gi} from Genbank...")
    # Fetch the record from the 'protein' database instead of 'nucleotide'
    try:
        net_handle = Entrez.efetch(db="protein", id=top_gi, rettype="fasta", retmode="text")
        seq_record = SeqIO.read(net_handle, "fasta")
        net_handle.close()

        print(f"\nSuccessfully downloaded sequence: {seq_record.id}")
        print(f"Description: {seq_record.description}")
        print(f"Sequence length: {len(seq_record.seq)} aa")
        print(f